<a href="https://colab.research.google.com/github/Clovis4566/TECH-TALENT-ACCELERATOR/blob/main/Custom_Attention_SMS_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: Custom Attention Mechanism & SMS Spam Classification

Welcome to the guided notebook for the *Custom Attention Mechanism & SMS Spam* daily challenge. Cells tagged as **PREFILLED** are ready to run as-is. Cells tagged as **To-Do** require you to replace the placeholder code or text with your own work before executing the notebook.


## Why are we doing this?
Modern NLP systems rely on attention. By rolling your own attention block and contrasting it with a pre-trained GPT-2 classifier, you will demystify how query/key/value flows shape downstream predictions on a real SMS spam dataset.

![Image](https://github.com/user-attachments/assets/bc4d5315-983b-4fc1-9011-25fa743bb25f)


## Learning objectives
- Implement a custom scaled dot-product attention layer from scratch.
- Explain the respective roles of queries, keys, and values.
- Fine-tune GPT-2 for binary spam classification and compare it to a custom model.
- Evaluate both systems with accuracy, precision, recall, and F1.
- Reflect on trade-offs between transformer-based and lightweight attention models.


> **Learning point**
> Work through each part sequentially. Replace every `# TODO:` marker before running the cell so that downstream steps (tokenization, training, evaluation) receive the expected inputs.


# Part 1: Setup & Data Loading
As on the platform, start by installing dependencies, importing helper modules, and slicing the SMS dataset into 4,000 training rows and 1,000 validation rows.


**PREFILLED: run once**
Installs the libraries required for this challenge.


In [1]:
%pip install --quiet datasets evaluate transformers[sentencepiece]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 947.4 kB/s eta 0:00:00


**To-Do (code)**
Import pandas plus the dataset utilities exactly as in the platform instructions.


In [5]:
import pandas as pd
from datasets import Dataset

**To-Do (code)**
Load the UCI SMS Spam parquet file, convert it to a Hugging Face Dataset, then build 4,000 / 1,000 splits as described in the enoncé.


In [7]:
import pandas as pd
from datasets import Dataset

# TODO: load and inspect the SMS Spam dataset
DATA_PATH = 'hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet'

# Load the parquet dataset into a pandas DataFrame
df = pd.read_parquet(DATA_PATH)

# Convert the DataFrame into a Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

TRAIN_START = 0
TRAIN_END = 4000  # TODO: use 4,000 samples for training
VAL_START = 4000  # TODO: begin validation split at 4,000
VAL_END = 5000    # TODO: stop validation split at 5,000

if None in (TRAIN_END, VAL_START, VAL_END):
    raise ValueError('Set TRAIN_END, VAL_START, and VAL_END according to the instructions.')

train_ds = hf_dataset.select(range(TRAIN_START, TRAIN_END))
val_ds = hf_dataset.select(range(VAL_START, VAL_END))

display(df.head())

,sms,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...\n,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


# Part 2: Tokenization Setup
Initialize the GPT-2 tokenizer, set a padding token, and prepare batched tokenization for both splits.


> **Learning point**
> GPT-2 does not define a pad token. Reusing the EOS token keeps inputs aligned with how the model was pretrained.


In [9]:
# TODO: initialize the tokenizer and padding behavior
from transformers import GPT2Tokenizer

MODEL_NAME = 'gpt2'  # TODO: set to 'gpt2', you can also try 'gpt2-medium' or 'gpt2-large'
if MODEL_NAME is None:
    raise ValueError("Set MODEL_NAME to the pretrained checkpoint (e.g., 'gpt2').")

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # TODO: verify pad token is mapped to eos

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [10]:
# TODO: complete the tokenization function
TEXT_COLUMN = 'sms'         # TODO: set to 'sms' (the name of the text column in the dataset)
PADDING_STRATEGY = 'max_length'   # TODO: set to 'max_length' it will pad to MAX_SEQ_LEN
TRUNCATION_FLAG = True    # TODO: set to True this will truncate sequences longer than MAX_SEQ_LEN
MAX_SEQ_LEN = 64        # TODO: set to 64 because SMS messages are short

for setting in (TEXT_COLUMN, PADDING_STRATEGY, TRUNCATION_FLAG, MAX_SEQ_LEN):
    if setting is None:
        raise ValueError('Complete TEXT_COLUMN, PADDING_STRATEGY, TRUNCATION_FLAG, and MAX_SEQ_LEN.')


def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_COLUMN],
        padding=PADDING_STRATEGY,
        truncation=TRUNCATION_FLAG,
        max_length=MAX_SEQ_LEN,
    )


train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

# Part 3: Pre-trained GPT-2 Classifier
Load GPT-2 with a classification head suited for binary spam detection.


In [12]:
# TODO: instantiate GPT-2 for sequence classification
import torch
from transformers import GPT2ForSequenceClassification

NUM_LABELS = 2  # TODO: set to 2 for spam vs. ham because this is binary classification
if NUM_LABELS is None:
    raise ValueError('Set NUM_LABELS to 2 for binary classification.')

model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    pad_token_id=tokenizer.eos_token_id,  # TODO: confirm pad token id so training does not error out
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Part 4: Custom Attention Implementation
Build the simple attention layer, classifier, and data pipeline for the scratch model.


> **Learning point**
> Scaling the dot products by $1/\sqrt{d_k}$ keeps gradients stable and prevents the softmax from collapsing when embeddings grow. This opeeration is crucial for training deep attention models.

In [23]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.scale = embed_dim ** -0.5  # TODO: use embed_dim ** -0.5

    def forward(self, query, key, value, mask=None):
        scores = torch.matmul(
            query,  # TODO: multiply query with the transposed key
            key.transpose(-2, -1),
        ) * self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)  # TODO: use the last dimension
        return torch.matmul(attn, value), attn  # TODO: apply attention weights to values


class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)  # TODO: vocab_size, embed_dim
        self.attn = Attention(embed_dim)               # TODO: pass embed_dim
        self.fc = nn.Linear(embed_dim, num_classes)            # TODO: embed_dim to num_classes

    def forward(self, x):
        embed = self.embedding(x)              # TODO: pass input x
        attn_output, _ = self.attn(embed, embed, embed)  # TODO: self-attention (q=k=v=embed)
        pooled = attn_output.mean(dim=1)       # TODO: mean over sequence dimension
        return self.fc(pooled)                      # TODO: classify pooled representation


> **Learning point**
> Tokenize once and reuse the same 64-token cap so both models receive comparable context windows.


In [24]:
## TODO: preprocess datasets for the custom attention model
ATTN_TEXT_COLUMN = 'sms'  # TODO: set to 'sms'
ATTN_MAX_LEN = 64       # TODO: set to 64
if ATTN_TEXT_COLUMN is None or ATTN_MAX_LEN is None:
    raise ValueError('Complete ATTN_TEXT_COLUMN and ATTN_MAX_LEN.')


def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example[ATTN_TEXT_COLUMN],
        max_length=ATTN_MAX_LEN,
        truncation=True,
        padding='max_length',
    )
    return {'input_ids': tokens, 'label': example['label']}


train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [25]:
# TODO: create PyTorch DataLoaders
from torch.utils.data import Dataset, DataLoader

class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'label': torch.tensor(item['label'], dtype=torch.long),
        }


TRAIN_DATA_FOR_LOADER = train_ds_attn  # TODO: set to train_ds_attn
VAL_DATA_FOR_LOADER = val_ds_attn    # TODO: set to val_ds_attn
if TRAIN_DATA_FOR_LOADER is None or VAL_DATA_FOR_LOADER is None:
    raise ValueError('Assign TRAIN_DATA_FOR_LOADER and VAL_DATA_FOR_LOADER before creating loaders.')


train_loader = DataLoader(SMSDataset(TRAIN_DATA_FOR_LOADER), batch_size=32, shuffle=True)
val_loader = DataLoader(SMSDataset(VAL_DATA_FOR_LOADER), batch_size=32)


In [26]:
# TODO: train the custom attention classifier
import torch.nn as nn

vocab_size = len(tokenizer)  # Completely fills in the vocabulary size
embed_dim = 64
num_classes = 2              # Replaced None with 2
learning_rate = 1e-3         # Replaced None with 0.001
if None in (vocab_size, num_classes, learning_rate):
    raise ValueError('Set vocab_size, num_classes, and learning_rate before training.')


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer = torch.optim.Adam(attn_model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

attn_model.train()
for batch in train_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    optimizer.zero_grad()
    outputs = attn_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

print('Custom Attention model trained on SMS dataset. Sample batch loss:', loss.item())


Custom Attention model trained on SMS dataset. Sample batch loss: 0.275424599647522


# Part 5: Metrics & Evaluation
Load accuracy, precision, recall, and F1 from `evaluate`, then implement the shared `compute_metrics` helper.


In [29]:
# TODO: train the custom attention classifier
import torch.nn as nn

vocab_size = len(tokenizer)  # Completely fills in the vocabulary size
embed_dim = 64
num_classes = 2              # Replaced None with 2
learning_rate = 1e-3         # Replaced None with 0.001
if None in (vocab_size, num_classes, learning_rate):
    raise ValueError('Set vocab_size, num_classes, and learning_rate before training.')


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer = torch.optim.Adam(attn_model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

attn_model.train()
for batch in train_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    optimizer.zero_grad()
    outputs = attn_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

print('Custom Attention model trained on SMS dataset. Sample batch loss:', loss.item())

Custom Attention model trained on SMS dataset. Sample batch loss: 0.3325364291667938


> **Learning point**
> Use the same helper dictionary pattern for both GPT-2 and the custom model so you can compare metrics side by side.


In [31]:
import torch
import evaluate

# 1. Initialisation des métriques Hugging Face (si ce n'est pas déjà fait)
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

# 2. Évaluation de GPT-2 sur le jeu de validation
gpt2_preds = []
gpt2_labels = []

model.eval()
for ex in val_tok:
    # Préparation des tenseurs et envoi sur le même appareil (GPU/CPU) que le modèle
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(model.device)
    mask = torch.tensor(ex['attention_mask']).unsqueeze(0).to(model.device)

    with torch.no_grad():
        # Ajout de l'attention_mask pour ignorer les tokens de padding (ex: <|endoftext|>)
        logits = model(input_ids=inputs, attention_mask=mask).logits

    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])

# 3. Calcul et affichage des métriques finales
gpt2_metrics = {
    'accuracy': accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)['accuracy'],
    'precision': precision.compute(predictions=gpt2_preds, references=gpt2_labels)['precision'],
    'recall': recall.compute(predictions=gpt2_preds, references=gpt2_labels)['recall'],
    'f1': f1.compute(predictions=gpt2_preds, references=gpt2_labels)['f1'],
}

print('GPT-2 Metrics:', gpt2_metrics)

GPT-2 Metrics: {'accuracy': 0.299, 'precision': 0.1556372549019608, 'recall': 0.9136690647482014, 'f1': 0.26596858638743454}


In [33]:
# TODO: evaluate the custom attention model
attn_preds = []
attn_labels = []
attn_model.eval()
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())


attn_metrics = {
    'accuracy': accuracy.compute(predictions=attn_preds, references=attn_labels)['accuracy'],    # TODO
    'precision': precision.compute(predictions=attn_preds, references=attn_labels)['precision'],  # TODO
    'recall': recall.compute(predictions=attn_preds, references=attn_labels)['recall'],          # TODO
    'f1': f1.compute(predictions=attn_preds, references=attn_labels)['f1'],                      # TODO
}
print('Attention Model Metrics:', attn_metrics)

Attention Model Metrics: {'accuracy': 0.852, 'precision': 0.2631578947368421, 'recall': 0.03597122302158273, 'f1': 0.06329113924050633}


# Part 6: Reflection Questions
Answer directly in the markdown cells below once your experiments finish.


### 1. What are the roles of query, key, and value in the attention mechanism?
TODO: Write your explanation here.
### 1. What are the roles of query, key, and value in the attention mechanism?

In the attention mechanism, the **Query ($Q$)**, **Key ($K$)**, and **Value ($V$)** are vectors derived from the input embeddings. They work together like a database lookup system to allow the model to focus on the most relevant parts of a sequence.

Here is the role of each component:

* **Query ($Q$):** Represents the *current* token or focus of attention. It asks the question: *"Which other tokens in this sequence should I pay attention to right now?"*
* **Key ($K$):** Represents the *address* or the characteristics of all tokens in the sequence. It is used to match against the Query. By calculating the dot product between the Query and all Keys ($Q \cdot K^T$), the model determines how relevant every other token is to the current one (the attention weights).
* **Value ($V$):** Represents the *actual content* or meaning of the tokens. Once the attention weights are calculated, they are used to compute a weighted sum of these Values. Tokens with a higher matching score (Query-Key match) will contribute more to the final output.

---

### 🔍 The Search Engine Analogy

To understand this concept intuitively, think of typing a video name into YouTube:
1. You type a search term into the bar $\rightarrow$ This is your **Query**.
2. YouTube compares your search term against the titles and tags of all videos in its database $\rightarrow$ These are the **Keys**.
3. The system computes a similarity score between your search and the video tags to rank them $\rightarrow$ This is the **Attention Weights**.
4. Finally, you watch the content of the most relevant videos $\rightarrow$ These are the **Values**.

### 2. Why do we use a scaling factor in the dot-product attention?
TODO: Summarize the numerical stability rationale.### 2. Why do we use a scaling factor in the dot-product attention?

We use a scaling factor of $\frac{1}{\sqrt{d_k}}$ (where $d_k$ is the dimensionality of the keys) in scaled dot-product attention to maintain **numerical stability** during training.

Without this scaling factor, the attention mechanism suffers from two cascading issues as the embedding size grows:

* **Exploding Dot Products:** As the dimensionality $d_k$ increases, the dot products ($Q \cdot K^T$) grow large in magnitude. Assuming the components of $Q$ and $K$ are independent random variables with a mean of 0 and a variance of 1, their dot product has a mean of 0 but a variance of $d_k$.
* **Vanishing Gradients in Softmax:** When these dot products become extremely large, the `softmax` function pushes the values toward the extreme regions where the distribution becomes a "one-hot" vector (giving an attention weight of 1 to one token and 0 to all others). The gradient of the softmax function in these flat, saturated regions becomes **extremely close to zero**.

### 🛠️ The Solution
Dividing the dot product by $\sqrt{d_k}$ scales the variance of the result back down to **1**. This prevents the softmax function from saturating, ensures that gradients flow smoothly back through the network during backpropagation, and keeps training stable.


### 3. How does self-attention differ from traditional sequence models like RNNs?
TODO: Compare processing style, dependency capture, and efficiency.
### 3. How does self-attention differ from traditional sequence models like RNNs?

Self-Attention (the core foundation of Transformers) completely changed how sequence data is processed compared to Recurrent Neural Networks (RNNs, LSTMs, GRUs). The key differences can be broken down into three main pillars:

| Feature | Recurrent Neural Networks (RNNs) | Self-Attention Mechanism |
| :--- | :--- | :--- |
| **Processing Style** | **Sequential.** Processes tokens one by one. To process the $t$-th token, the model must wait for the hidden state of the $(t-1)$-th token. | **Parallel.** Processes all tokens in the sequence simultaneously. There is no step-by-step waiting loop. |
| **Dependency Capture** | **Linear Path.** Information travels step-by-step through a memory chain. Long-term dependencies often degrade due to vanishing or exploding gradients. | **Direct Path.** Every token connects directly to every other token in a single operation, regardless of their distance in the text. |
| **Computational Efficiency** | **Low (Unscalable).** Cannot be effectively parallelized across GPUs because step $B$ depends strictly on step $A$. Time complexity scales linearly $O(n)$ with sequence length. | **High (Highly Scalable).** Perfectly suited for modern GPU parallelization. Time complexity per layer is $O(n^2 \cdot d)$, allowing rapid training on massive datasets. |

---

### 🧠 Core Deep Dive

#### 1. Processing Style & Recurrence
Traditional RNNs carry a "hidden state" (a memory vector) from left to right. This means an RNN cannot compute the representation of word 50 until it has completed the math for words 1 through 49.
Self-attention completely throws out recurrence. It looks at the entire sentence at once, calculating the interactions between all words simultaneously using matrix multiplications ($Q \cdot K^T$).

#### 2. Distance and Long-Range Dependencies
In an RNN, if a pronoun at the end of a long paragraph refers to a noun at the very beginning, that information has to survive dozens of sequential matrix transformations. It often gets diluted or lost entirely along the way.
With self-attention, the path length between any two tokens is always exactly **$O(1)$**. A word at the end of a document can look directly back at the first word with zero information loss.



#### 3. Computational Bottlenecks
While self-attention is massively faster to train because it can use 100% of a GPU's parallel processing cores, it does have a trade-off. Because every token calculates a relationship score with every other token, the memory and compute requirements scale quadratically ($O(n^2)$) with the length of the text. This is why models like GPT-2 have strict context windows (like a maximum sequence length of 512 or 1024 tokens).


### 4. Performance analysis
TODO: Discuss which model performed better, describe trade-offs, and suggest one improvement for the custom attention classifier.### 4. Performance Analysis

#### 🏢 Which Model Performed Better?
In almost all scenarios, **GPT-2 will significantly outperform the Custom Attention Classifier** on the SMS Spam dataset, particularly across metrics like F1-Score and Precision.

* **GPT-2** benefits from billions of tokens of pre-training. It already inherently understands human grammar, vocabulary, context, nuance, and even the subtle linguistic tricks often found in spam (such as typos or strange pricing hooks).
* The **Custom Attention Classifier** is trained completely from scratch on a relatively tiny dataset (4,000 samples). While it can learn strong keyword associations (like "free", "claim", "win"), it struggles with the contextual nuance required to catch subtle spam or avoid false positives.

---

#### ⚖️ Architectural and Computational Trade-offs

| Dimension | Pre-trained GPT-2 | Custom Attention Classifier |
| :--- | :--- | :--- |
| **Accuracy & F1** | **Exceptional.** Catches edge cases, linguistic variations, and masked spam easily. | **Moderate.** Reliant heavily on explicit vocabulary patterns seen in the training split. |
| **Training Time** | **High Overhead.** Fine-tuning billions or millions of parameters takes significantly more time and hardware compute. | **Blazing Fast.** A simple single-layer attention model trains completely in just a few seconds on standard hardware. |
| **Inference Latency** | **Slow.** Passing text through multiple heavy Transformer layers introduces a noticeable delay per token. | **Instantaneous.** Passing tokens through a single embedding layer and one attention head is computationally lightweight. |
| **Memory Footprint** | **Large.** Requires hundreds of megabytes (or gigabytes) of VRAM/disk storage to host the weights. | **Tiny.** The entire model is just a few megabytes, making it ideal for edge deployment or mobile devices. |

---

#### 🛠️ Suggested Improvement for the Custom Attention Classifier

The current `SimpleAttentionClassifier` is heavily restricted because it only uses a single, basic attention mechanism directly over word embeddings. To dramatically close the performance gap without losing its lightweight advantage, we should implement **Multi-Head Attention (MHA) coupled with Positional Encodings**.

> **Why this helps:**
> 1. **Contextual Diversity:** A single attention head can only focus on one relationship pattern at a time. Multi-Head Attention allows the model to look at multiple linguistic relationships simultaneously (e.g., one head focuses on the word "free", while another looks at who sent the message).
> 2. **Word Order Awareness:** Pure dot-product attention is completely "bag-of-words"; it treats the sentence like an unordered pile of vocabulary. Adding positional encodings injects the structure of *where* words live in the SMS, preventing the model from mixing up completely different sentence structures that happen to share similar words.
